# 📘 BERT 논문 분석

> **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**  
> Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova (Google AI Language, 2019)  
> [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

비전공자도 이해할 수 있도록 정리한 BERT 논문 학습 노트입니다.

---

## 📑 목차

1. Abstract 3줄 요약
2. Introduction + Conclusion
3. 핵심 구조 설명
4. 구조 시각화
5. 논문 ↔ 코드 연결 (PyTorch)
6. 데이터 흐름 추적 (Shape 변화)

## 1. Abstract 3줄 요약

- **문제:** <ins>기존 언어모델(GPT, ELMo)은 단방향이거나 얕은 양방향</ins>이라, <ins>양쪽 문맥을 깊이 이해해야 하는 NLP 태스크에서 성능이 제한</ins>된다.
- **방법:** <ins>입력 문장의 일부 단어를 가리고(Masked LM) 그 단어를 맞추는 방식</ins> + <ins>두 문장이 이어지는지 맞추는(NSP) 방식</ins>으로, <ins>양방향 Transformer Encoder를 사전학습</ins>한다.
- **결과:** <ins>단 하나의 출력 레이어만 추가해 파인튜닝</ins>하면, <ins>11개 NLP 태스크에서 모두 SOTA 달성</ins> (GLUE 80.5%, SQuAD v1.1 F1 93.2, SQuAD v2.0 F1 83.1).

## 2. Introduction + Conclusion

### 🤔 왜 이 논문이 나왔는가?

당시 NLP에는 사전학습된 언어모델을 활용하는 두 가지 방식이 있었다.

- **Feature-based (ELMo):** <ins>사전학습된 표현을 "추가 피처"로만 사용</ins>. 태스크마다 별도의 복잡한 모델 구조를 만들어야 함.
- **Fine-tuning (GPT):** <ins>사전학습된 모델 전체</ins>를 다운스트림 태스크에 맞춰 <ins>미세조정</ins>. 더 간단함.

### ❌ 기존 방식의 한계

**가장 큰 문제는 "단방향성"이다.**

- GPT는 왼쪽 → 오른쪽으로만 문맥을 본다. `"The cat sat on the ___"`에서 ___ 다음 단어를 볼 수 없음.
- ELMo는 왼쪽→오른쪽과 오른쪽→왼쪽 모델을 각각 학습한 뒤 "얕게(shallow)" 이어 붙임. 진짜 양방향이 아님.

**이게 왜 문제인가?** <ins>질의응답(QA), 개체명 인식(NER) 같은 토큰 단위 태스크에서는 단어의 앞뒤 문맥을 모두 봐야 정답을 알 수 있다.</ins>

### ✅ BERT의 개선점

1. **진짜 양방향 학습** → "Masked Language Model (MLM)"이라는 <ins>빈칸 채우기 방식</ins>을 도입해서, <ins>모든 레이어에서 좌우 문맥을 동시에 참고</ins>할 수 있게 함.
2. **문장 관계 학습** → "Next Sentence Prediction (NSP)" 태스크로 <ins>두 문장의 관계까지 학습</ins>.
3. **통일된 구조** → 사전학습과 파인튜닝의 모델 구조가 거의 동일. <ins>출력 레이어 하나만 갈아 끼우면 됨</ins>.

> 💡 BERT는 "양방향 사전학습이 진짜로 효과 있다"는 걸 11개 태스크에서 SOTA로 증명한 논문이며, 이후 NLP의 패러다임을 바꿔놓았다.

## 3. 핵심 구조 설명

### 🧩 구성 요소 1. WordPiece Tokenizer

> **비유:** 단어를 통째로 외우지 않고 "조각(piece)"으로 쪼개서 외우는 방식. 영어 단어 `"playing"`을 `"play"`와 `"##ing"`으로 나누는 식.

| 항목 | 설명 |
|---|---|
| **이게 뭔지** | 입력 텍스트를 미리 정한 30,000개의 토큰 단위로 쪼개는 도구 |
| **입력값** | 문자열 (예: `"my dog is playing"`) |
| **출력값** | 토큰 ID 리스트 (예: `[101, 2026, 3899, 2003, 2652, 102]`) |
| **하이퍼파라미터** | 어휘 크기(vocab size = 30,000) |

### 🧩 구성 요소 2. Input Embedding (3가지 임베딩의 합)

> **비유:** 단어 하나를 표현할 때 "이 단어가 뭔지(Token) + 어느 문장인지(Segment) + 몇 번째 위치인지(Position)"를 모두 더해서 하나의 벡터로 만든다.

| 항목 | 설명 |
|---|---|
| **이게 뭔지** | 토큰을 모델이 이해할 수 있는 벡터로 바꾸는 단계 (세 임베딩을 더해서 만듦) |
| **입력값** | 토큰 ID 리스트 `(batch, seq_len)` |
| **출력값** | 임베딩 텐서 `(batch, seq_len, H)` |
| **하이퍼파라미터** | Hidden size(H), Max sequence length(512), Vocab size(30,000), Segment 개수(2) |

**세 가지 임베딩:**
- **Token Embedding:** 단어의 의미
- **Segment Embedding:** 첫 번째 문장(A)인지 두 번째 문장(B)인지
- **Position Embedding:** 시퀀스 내 위치 (BERT는 학습 가능한 position embedding을 씀, sinusoidal 아님)

### 🧩 **구성 요소 3. Transformer Encoder (BERT의 본체)**

> **비유:** 한 단어가 문장 안의 다른 모든 단어들에게 "너랑 나랑 얼마나 관련 있어?"라고 물어보고, 관련 깊은 단어의 정보를 더 많이 가져와서 자기 자신을 업데이트하는 회의실.

| 항목 | 설명 |
|---|---|
| **이게 뭔지** | Self-Attention으로 각 토큰이 양방향으로 모든 토큰을 참조하며 문맥 정보를 학습하는 블록 |
| **입력값** | `(batch, seq_len, H)` |
| **출력값** | `(batch, seq_len, H)` (shape 동일, 의미만 풍부해짐) |
| **하이퍼파라미터** | L (블록 수), H (hidden size), A (attention head 수), FFN size = 4H, GELU, Dropout 0.1 |

| 모델 | L | H | A | 파라미터 |
|---|---|---|---|---|
| BERT-BASE | 12 | 768 | 12 | 110M |
| BERT-LARGE | 24 | 1024 | 16 | 340M |

> ⚠️ **핵심 포인트:** GPT의 Transformer는 "마스킹된 self-attention"(왼쪽만 봄), **BERT는 "양방향 self-attention"(전체를 봄)**. 이게 두 모델의 결정적 차이다.

### 🧩 구성 요소 4. Masked Language Model (MLM) — **(핵심)**

> **비유:** 영어 시험에서 빈칸 채우기 문제. `"The cat ___ on the mat"` → `"sat"`을 맞추라는 것. 단, BERT는 문장 전체를 다 보면서 빈칸을 채울 수 있다(양방향).

| 항목 | 설명 |
|---|---|
| **이게 뭔지** | 입력 토큰 중 15%를 무작위로 가리고, 모델이 원래 단어를 맞추도록 학습하는 사전학습 태스크 |
| **입력값** | 일부 토큰이 가려진 시퀀스 `(batch, seq_len)` |
| **출력값** | 가려진 위치에서 vocab 전체에 대한 확률 분포 `(batch, num_masked, vocab_size=30000)` |
| **하이퍼파라미터** | 마스킹 비율 15%, 마스킹 전략 80/10/10, Cross Entropy Loss |

**80/10/10 전략 (선택된 15% 안에서):**
- 80%는 `[MASK]` 토큰으로 교체
- 10%는 랜덤 단어로 교체
- 10%는 원래 단어 유지

> 🤔 **왜 이런 복잡한 전략을 쓰나?**  
> 사전학습 때만 `[MASK]` 토큰이 등장하고 파인튜닝 때는 없다. 이 "불일치"를 줄이려고, 모든 토큰이 "혹시 내가 예측 대상일지 모른다"는 자세를 유지하게끔 만든 것.

### 🧩 구성 요소 5. Next Sentence Prediction (NSP) — **(핵심)**

> **비유:** 두 문단을 보여주고 "이 두 문단이 원래 이어지는 글이었어?"라고 묻는 OX 퀴즈. QA나 자연어 추론처럼 두 문장 사이 관계를 봐야 하는 태스크에서 도움이 된다.

| 항목 | 설명 |
|---|---|
| **이게 뭔지** | 두 문장(A, B)이 실제로 연속인지(IsNext) 무작위 쌍인지(NotNext) 맞추는 이진 분류 태스크 |
| **입력값** | `[CLS] 문장A [SEP] 문장B [SEP]` 형태의 시퀀스 |
| **출력값** | `[CLS]` 위치의 벡터를 이용한 IsNext/NotNext 이진 분류 `(batch, 2)` |
| **하이퍼파라미터** | 긍정/부정 샘플 비율 50:50, Cross Entropy Loss (MLM loss와 합산) |

---

### 🧩 구성 요소 6. Special Tokens

| 토큰 | 비유 | 역할 |
|---|---|---|
| `[CLS]` | 회의의 "요약 담당자" | 시퀀스 맨 앞. 마지막 레이어의 `[CLS]` 벡터(C ∈ ℝ^H)는 분류 태스크의 대표값 |
| `[SEP]` | "여기까지가 한 문장" 표지판 | 문장 사이 구분자 |
| `[MASK]` | 빈칸 채우기의 `___` | MLM 학습용 마스킹 토큰 |

---

### 🧩 구성 요소 7. Pre-training vs Fine-tuning

> **비유:** Pre-training은 "초중고 일반 교육"이고, Fine-tuning은 "대학에서 전공 공부". 같은 사람(같은 파라미터)이 일반 지식을 쌓은 뒤, 특정 분야로 특화되는 것.

| 단계 | 데이터 | 학습 시간 |
|---|---|---|
| **Pre-training** | BooksCorpus(800M) + Wikipedia(2,500M) = 33억 단어 | 1M step, 4일 (TPU 4~16개) |
| **Fine-tuning** | 태스크별 라벨 데이터 | 1시간 ~ 몇 시간 (단일 GPU/TPU) |

> 💡 **이게 BERT의 진짜 매력:** 비싼 사전학습은 구글이 한 번만 하고, 우리는 가벼운 파인튜닝만으로 SOTA 모델을 만들 수 있다.

## 4. 구조 시각화

```mermaid
flowchart TD
    A["1. 원시 입력 Raw Text<br/>'my dog is cute'"] --> B["2. WordPiece Tokenizer<br/>(batch, seq_len) — 토큰 ID"]
    B --> C1["Token Embedding<br/>(B, L, H)"]
    B --> C2["Segment Embedding<br/>(B, L, H)"]
    B --> C3["Position Embedding<br/>(B, L, H)"]
    C1 -->|합산 Sum| D["3. 입력 임베딩 E<br/>(batch, seq_len, H=768)"]
    C2 -->|합산 Sum| D
    C3 -->|합산 Sum| D
    D --> E["4. Transformer Encoder × L층<br/>Multi-Head Self-Attention 양방향<br/>Add+LayerNorm → FFN 4H → Add+LayerNorm<br/>BASE: L=12, H=768, A=12"]
    E --> F["5. 최종 Hidden States<br/>C CLS벡터 + T₁..Tₙ 토큰 벡터"]
    F --> G1["6-A. Pre-training Heads<br/>MLM: Tᵢ → softmax vocab=30K<br/>NSP: C → softmax 2"]
    F --> G2["6-B. Fine-tuning Heads<br/>문장분류: C → softmax K<br/>NER: Tᵢ → softmax tags<br/>QA: Tᵢ·S, Tᵢ·E → start/end"]
    G1 --> H1["MLM Loss + NSP Loss<br/>라벨 없는 대량 텍스트"]
    G2 --> H2["태스크별 예측 결과<br/>라벨 있는 다운스트림 데이터"]
```

> 📌 **참고:** mermaid 다이어그램은 GitHub, VS Code, JupyterLab(확장 필요)에서 렌더링됩니다. 일반 Jupyter Notebook에서는 텍스트 코드만 보일 수 있습니다.

## 5. 논문 ↔ 코드 연결 (PyTorch)

### 📍 논문 표현 ↔ 코드 매핑

| 논문 표현 | PyTorch 코드 |
|---|---|
| "WordPiece embeddings with 30,000 vocab" | `nn.Embedding(30000, hidden_size)` |
| "Token + Segment + Position embedding을 합산" | `token_emb + segment_emb + position_emb` |
| "Multi-layer bidirectional Transformer encoder" | `nn.TransformerEncoder(layer, num_layers=L)` |
| "Multi-head self-attention" | `nn.MultiheadAttention(H, num_heads=A)` |
| "FFN size = 4H" | `nn.Linear(H, 4*H) → GELU → nn.Linear(4*H, H)` |
| "GELU activation instead of ReLU" | `nn.GELU()` |
| "Dropout probability = 0.1" | `nn.Dropout(0.1)` |
| "Final hidden of [CLS] is C ∈ ℝ^H" | `pooled_output = hidden_states[:, 0, :]` |
| "MLM: predict masked tokens over vocab" | `nn.Linear(H, vocab_size)` → CrossEntropyLoss |
| "NSP: binary classification with C" | `nn.Linear(H, 2)` → CrossEntropyLoss |
| "Classification: log(softmax(C·Wᵀ))" | `nn.Linear(H, num_classes)` |

---

### 💻 전체 모델 PyTorch 구현

아래 셀들을 순서대로 실행하면 BERT 미니 버전을 직접 만들고 동작시켜볼 수 있습니다.

### Step 1. 라이브러리 import

In [1]:
import torch
import torch.nn as nn

# 재현성을 위한 시드 설정
torch.manual_seed(42)

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

PyTorch 버전: 2.10.0+cpu
CUDA 사용 가능: False


### Step 2. `BertEmbeddings` — 3가지 임베딩 합산

Token + Segment + Position 임베딩을 모두 더해서 입력 벡터를 만든다.

In [2]:
class BertEmbeddings(nn.Module):
    """3가지 임베딩을 합산: Token + Segment + Position"""
    def __init__(self, vocab_size=30000, hidden_size=768,
                 max_len=512, num_segments=2):
        super().__init__()
        self.token_emb    = nn.Embedding(vocab_size, hidden_size)
        self.segment_emb  = nn.Embedding(num_segments, hidden_size)
        self.position_emb = nn.Embedding(max_len, hidden_size)
        self.layer_norm   = nn.LayerNorm(hidden_size)
        self.dropout      = nn.Dropout(0.1)

    def forward(self, input_ids, segment_ids):
        # input_ids:   (batch, seq_len)
        # segment_ids: (batch, seq_len)
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)

        emb = (self.token_emb(input_ids)
             + self.segment_emb(segment_ids)
             + self.position_emb(positions))   # (B, L, H)
        return self.dropout(self.layer_norm(emb))


# 동작 확인
emb_layer = BertEmbeddings()
test_ids = torch.randint(0, 30000, (2, 10))
test_seg = torch.zeros(2, 10, dtype=torch.long)
out = emb_layer(test_ids, test_seg)
print(f"입력 shape:  {test_ids.shape}")
print(f"출력 shape:  {out.shape}  # (batch=2, seq=10, H=768)")

입력 shape:  torch.Size([2, 10])
출력 shape:  torch.Size([2, 10, 768])  # (batch=2, seq=10, H=768)


### Step 3. `BertLayer` — Transformer Encoder 1개 블록

Self-Attention → Add & Norm → FFN → Add & Norm 순서로 구성된다.  
⚠️ FFN의 활성화 함수는 ReLU가 아닌 **GELU**임에 주의.

In [3]:
class BertLayer(nn.Module):
    """Transformer Encoder 1개 블록: Self-Attention + FFN"""
    def __init__(self, hidden_size=768, num_heads=12, ffn_size=3072):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            hidden_size, num_heads, dropout=0.1, batch_first=True
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),                 # ⚠️ ReLU 아님!
            nn.Linear(ffn_size, hidden_size),
            nn.Dropout(0.1),
        )
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x, attention_mask=None):
        # Self-Attention (양방향: 모든 토큰이 모든 토큰을 봄)
        attn_out, _ = self.attention(x, x, x, key_padding_mask=attention_mask)
        x = self.norm1(x + attn_out)           # Add & Norm

        # Feed-Forward Network
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)            # Add & Norm
        return x


# 동작 확인
layer = BertLayer()
test_input = torch.randn(2, 10, 768)
test_output = layer(test_input)
print(f"입력 shape:  {test_input.shape}")
print(f"출력 shape:  {test_output.shape}  # shape 유지!")

입력 shape:  torch.Size([2, 10, 768])
출력 shape:  torch.Size([2, 10, 768])  # shape 유지!


### Step 4. `BERT` — 전체 모델 조립

Embedding → Transformer 블록 L개 → MLM Head + NSP Head

In [4]:
class BERT(nn.Module):
    """BERT-Base: L=12, H=768, A=12"""
    def __init__(self, vocab_size=30000, hidden_size=768,
                 num_layers=12, num_heads=12, ffn_size=3072):
        super().__init__()
        self.embeddings = BertEmbeddings(vocab_size, hidden_size)
        self.encoder = nn.ModuleList([
            BertLayer(hidden_size, num_heads, ffn_size)
            for _ in range(num_layers)
        ])

        # Pre-training Heads
        self.mlm_head = nn.Linear(hidden_size, vocab_size)  # MLM
        self.nsp_head = nn.Linear(hidden_size, 2)           # NSP

    def forward(self, input_ids, segment_ids, attention_mask=None):
        # 1. Embedding
        x = self.embeddings(input_ids, segment_ids)         # (B, L, H)

        # 2. Transformer 블록 L번 통과
        for layer in self.encoder:
            x = layer(x, attention_mask)                    # (B, L, H)

        # 3. 출력
        C = x[:, 0, :]                # [CLS] 벡터: (B, H)
        T = x                         # 토큰별 벡터: (B, L, H)

        mlm_logits = self.mlm_head(T) # (B, L, vocab_size)
        nsp_logits = self.nsp_head(C) # (B, 2)
        return mlm_logits, nsp_logits, C, T


# 파라미터 수 확인
model = BERT()
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")
print(f"(논문상 BERT-Base는 110M, 우리 구현은 약간 다름)")

전체 파라미터 수: 131,562,290
(논문상 BERT-Base는 110M, 우리 구현은 약간 다름)


### Step 5. 실제로 돌려보기

batch 2개, 길이 128짜리 입력을 넣어 출력 shape를 확인하자.

In [5]:
# 더미 입력
input_ids   = torch.randint(0, 30000, (2, 128))   # (batch=2, seq=128)
segment_ids = torch.zeros(2, 128, dtype=torch.long)

# Forward pass
model.eval()
with torch.no_grad():
    mlm_logits, nsp_logits, C, T = model(input_ids, segment_ids)

print("=" * 50)
print(f"입력 input_ids:     {input_ids.shape}")
print(f"입력 segment_ids:   {segment_ids.shape}")
print("-" * 50)
print(f"MLM logits:         {mlm_logits.shape}  # (B, L, vocab=30000)")
print(f"NSP logits:         {nsp_logits.shape}  # (B, 2)")
print(f"C (CLS 벡터):       {C.shape}            # (B, H=768)")
print(f"T (전체 토큰 벡터): {T.shape}    # (B, L, H)")
print("=" * 50)

입력 input_ids:     torch.Size([2, 128])
입력 segment_ids:   torch.Size([2, 128])
--------------------------------------------------
MLM logits:         torch.Size([2, 128, 30000])  # (B, L, vocab=30000)
NSP logits:         torch.Size([2, 2])  # (B, 2)
C (CLS 벡터):       torch.Size([2, 768])            # (B, H=768)
T (전체 토큰 벡터): torch.Size([2, 128, 768])    # (B, L, H)


## 6. 데이터 흐름 추적 (Shape 변화)

**가정:** batch=2, seq_len=128, BERT-Base (H=768, L=12, A=12, vocab=30,000)

```
입력 텍스트 "[CLS] my dog is cute [SEP]"
           ↓ WordPiece Tokenizer
input_ids:    (2, 128)          # 정수 ID
segment_ids:  (2, 128)          # 0 또는 1

           ↓ [Embedding Layer] Token+Segment+Position을 합산
embeddings:   (2, 128, 768)     # 각 토큰이 768차원 벡터로 변환됨

           ↓ [Transformer Block 1] Self-Attention + FFN
hidden:       (2, 128, 768)     # shape 동일, 의미만 풍부해짐 (양방향 문맥 반영)

           ↓ [Transformer Block 2~12] 동일한 연산 11번 반복
hidden:       (2, 128, 768)     # 깊은 양방향 표현 학습

           ↓ [최종 Hidden State 분리]
C (CLS벡터):  (2, 768)          # hidden[:, 0, :] → 문장 전체 대표
T (토큰벡터): (2, 128, 768)     # hidden 전체 → 토큰별 의미

           ↓ [MLM Head]  T를 vocab 공간으로 사영
mlm_logits:   (2, 128, 30000)   # 각 위치마다 30000개 단어 확률

           ↓ [NSP Head]  C를 이진 분류 공간으로 사영
nsp_logits:   (2, 2)            # IsNext/NotNext 두 클래스
```

### 📍 각 단계 한 줄 설명

| 변화 | 이유 |
|---|---|
| `(2,128) → (2,128,768)` | 정수 토큰 ID가 768차원의 의미 벡터로 변환됨 (lookup table) |
| `(2,128,768) → (2,128,768)` × 12회 | Self-Attention으로 토큰 간 관계 학습. shape는 유지, 의미는 풍부해짐 |
| `(2,128,768) → (2,768)` | `[CLS]` 위치(0번 인덱스)만 뽑아냄. 문장 전체를 요약하는 벡터 |
| `(2,128,768) → (2,128,30000)` | 각 토큰 위치에서 vocab 30,000개 중 어떤 단어인지 예측 (MLM) |
| `(2,768) → (2,2)` | `[CLS]` 벡터로 두 문장이 이어지는지 이진 분류 (NSP) |

### 📍 파인튜닝 시 출력 변화

| 태스크 | 사용 벡터 | 변환 | 최종 shape |
|---|---|---|---|
| **문장 분류 (SST-2)** | C: `(B, 768)` | `Linear(768, 2)` | `(B, 2)` |
| **NER** | T: `(B, L, 768)` | `Linear(768, tags)` | `(B, L, tags)` |
| **QA (SQuAD)** | T: `(B, L, 768)` | `S·Tᵢ`, `E·Tᵢ` | `(B, L)`, `(B, L)` (start/end 확률) |
| **MNLI** | C: `(B, 768)` | `Linear(768, 3)` | `(B, 3)` |

### 💡 위 표를 실제로 코드로 확인하기

각 단계의 shape 변화를 직접 출력해보자.

In [6]:
# 중간 출력을 모두 확인하기 위해 단계별로 실행
model.eval()
with torch.no_grad():
    # Step 1: Embedding
    x = model.embeddings(input_ids, segment_ids)
    print(f"[Embedding 후]     {x.shape}")

    # Step 2: Transformer 블록 통과
    for i, layer in enumerate(model.encoder):
        x = layer(x)
        if i in [0, 5, 11]:  # 첫 번째, 중간, 마지막만 출력
            print(f"[Transformer L{i+1:>2}]   {x.shape}")

    # Step 3: 분리
    C = x[:, 0, :]
    T = x
    print(f"[CLS 벡터 C]       {C.shape}")
    print(f"[전체 토큰 T]      {T.shape}")

    # Step 4: Head 통과
    mlm = model.mlm_head(T)
    nsp = model.nsp_head(C)
    print(f"[MLM logits]      {mlm.shape}")
    print(f"[NSP logits]      {nsp.shape}")

[Embedding 후]     torch.Size([2, 128, 768])
[Transformer L 1]   torch.Size([2, 128, 768])
[Transformer L 6]   torch.Size([2, 128, 768])
[Transformer L12]   torch.Size([2, 128, 768])
[CLS 벡터 C]       torch.Size([2, 768])
[전체 토큰 T]      torch.Size([2, 128, 768])
[MLM logits]      torch.Size([2, 128, 30000])
[NSP logits]      torch.Size([2, 2])


---
## 🎯 최종 한 줄 요약

> **BERT는 "양방향 Transformer Encoder"를 "Masked LM + NSP"로 사전학습한 모델이며, 파인튜닝 시 출력 레이어 하나만 추가하면 다양한 NLP 태스크에서 SOTA를 달성한다.** 이 패러다임(Pre-training → Fine-tuning)이 이후 모든 NLP 연구의 표준이 되었다.

면접에서 BERT를 설명할 때는 다음 네 가지를 막힘없이 말할 수 있으면 충분하다:

1. 양방향성의 의미
2. MLM의 80/10/10 트릭
3. `[CLS]`와 `[SEP]`의 역할
4. Pre-training / Fine-tuning 분리의 장점

화이팅! 💪

---

## 📚 참고 자료

- [BERT 논문 (arXiv)](https://arxiv.org/abs/1810.04805)
- [공식 코드 (GitHub)](https://github.com/google-research/bert)
- [HuggingFace Transformers](https://huggingface.co/docs/transformers/model_doc/bert)
- [The Annotated Transformer](http://nlp.seas.harvard.edu/2018/04/03/attention.html)